# DROID ground truth — a self-contained Colab notebook

DROID is the robot-manipulation data source of TAPVid-MV: a Franka arm working on a tabletop,
filmed by a **heterogeneous three-camera rig** — one **wrist camera riding on the arm** plus **two
fixed exterior Zed cameras** — with stereo depth, a wrist-view foreground mask, and 400–600 3D
point tracks per sequence in one world frame.

This notebook needs **no repository checkout and no credentials** — every array is read straight
from the public bucket `storage.googleapis.com/dm-tapnet/mv-tap`.

| File | Meaning |
| --- | --- |
| `tracks_xyz.npy` `(T, N, 3)` | 3D tracks in **world coordinates**; all three views share the point identities |
| | `T` is 53–244 frames depending on the episode, `N` is 400–600 tracks |
| `queries_xytv.npy` `(N, 4)` | the query per track: `(x, y, t, view)`, spread over all three views |
| `<v>/intrinsics.npy` `(4,)` | `(fx, fy, cx, cy)`, integer-centre pixel convention |
| `<v>/extrinsics_w2c.npy` `(T, 4, 4)` | per-frame world-to-camera; **only view 0 actually moves** |
| `<v>/visibility.npy` `(T, N)` | visibility per view — the cross-view occlusion structure |
| `<v>/images_jpeg_bytes.npy` `(T,)` | JPEG bytes, one per frame, 1280x720 |
| `<v>/depth.npy` `(T, 720, 1280)` | stereo depth in metres |
| `0/foreground_mask.npy` | **wrist view only** — the two exterior views ship no mask |

**What makes DROID different from the other sources**: the rig is *mixed*. View 0 is bolted to the
robot and sweeps through the scene while views 1 and 2 never move, so "the point moved" and "the
camera moved" pull apart in a way no other data source in the benchmark shows. Sections 3 and 7
are about exactly that.

**The 2 m depth policy**: DROID's stereo depth is only trustworthy inside the workspace, so the
benchmark zeroes everything beyond `MAX_TRACKING_DEPTH_M = 2.0` before handing depth to a tracker.
Section 7 shows the raw map and the capped one side by side.

**Download size**: a sequence is up to 2 GB in the bucket, but nearly all of that is depth and
masks. This notebook fetches only the light arrays (**~70 MB**, almost all JPEG frames) and reads
depth / masks **one frame at a time over HTTP range requests**.


In [ ]:
#@title Setup — install and import
!pip install -q mediapy

import functools
import io
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import mediapy
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
print("ready")


## 1. The reader

Plain NumPy / OpenCV, no repository dependency.

Two DROID-specific wrinkles are handled here: `foreground_mask.npy` exists **only for view 0**, so
`View.foreground_mask` returns `None` for the exterior cameras and nothing is requested for them;
and `View.depth(frame, capped=True)` applies the benchmark's 2 m workspace policy.

`RemoteFrames` is the piece worth reading: a DROID depth file is up to 700 MB, so the `.npy`
header is parsed once out of the first 4 KB and each frame afterwards is a single 3.7 MB byte
range. Set `DOWNLOAD_DEPTH = True` below to hold the files locally instead and the reader
transparently switches to a memmap.


In [ ]:
#@title The DROID reader (run once)
BUCKET = "https://storage.googleapis.com/dm-tapnet/mv-tap/droid/tapvidmv"


def find_local_dataset(name="droid"):
    """Return an already-downloaded dataset next to this notebook, if there is one.

    On Colab there is nothing on disk and everything is fetched from the bucket.
    Run the same notebook inside a repository checkout and this finds the
    ``tapvidmv_dataset/<name>`` you already have, so not a byte is downloaded.
    """
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "tapvidmv_dataset" / name
        if candidate.is_dir():
            return candidate
    return None


DATA_ROOT = find_local_dataset() or Path("droid_data")
print(f"reading from {DATA_ROOT}"
      f"{'' if find_local_dataset() else ' (will download from the bucket)'}")

# The benchmark masks DROID depth outside a 2 m workspace: the Zed stereo depth
# degrades badly past the table, so anything beyond this is not tracker input.
MAX_TRACKING_DEPTH_M = 2.0

SEQUENCES = [
    "AUTOLab+0d4edc83+2023-10-21-19h-45m-29s",
    "AUTOLab+0d4edc83+2023-12-02-13h-00m-54s",
    "AUTOLab+0d4edc83+2023-12-02-13h-49m-51s",
    "AUTOLab+0d4edc83+2023-12-02-15h-25m-11s",
    "AUTOLab+0d4edc83+2023-12-02-15h-34m-31s",
    "AUTOLab+5d05c5aa+2023-07-23-21h-12m-36s",
    "AUTOLab+5d05c5aa+2023-08-12-20h-50m-04s",
    "AUTOLab+7cb094d5+2023-12-13-16h-09m-15s",
    "AUTOLab+84bd5053+2023-07-21-15h-36m-07s",
    "AUTOLab+84bd5053+2023-07-27-08h-56m-33s",
    "AUTOLab+84bd5053+2023-07-27-09h-03m-35s",
    "AUTOLab+84bd5053+2023-07-27-09h-05m-54s",
    "AUTOLab+84bd5053+2023-10-13-19h-35m-42s",
    "AUTOLab+9aed6d7a+2023-12-02-17h-46m-32s",
    "AUTOLab+cf2a60c6+2023-11-21-09h-59m-22s",
    "CLVR+13759f6e+2023-05-16-14h-58m-30s",
    "CLVR+13759f6e+2023-05-17-21h-25m-32s",
    "CLVR+13759f6e+2023-05-20-18h-03m-28s",
    "CLVR+236539bc+2023-05-09-04h-35m-06s",
    "CLVR+236539bc+2023-05-16-19h-40m-14s",
    "CLVR+236539bc+2023-05-17-19h-21m-00s",
    "CLVR+236539bc+2023-06-25-18h-23m-03s",
    "GuptaLab+553d1bd5+2023-05-19-11h-00m-57s",
    "ILIAD+7ae1bcff+2023-05-23-19h-07m-29s",
    "IPRL+edf28ef3+2023-12-19-09h-19m-31s",
    "IRIS+7dfa2da3+2023-05-05-11h-00m-04s",
    "IRIS+7dfa2da3+2023-11-02-15h-42m-41s",
    "IRIS+ef107c48+2023-03-02-16h-37m-19s",
    "PennPAL+06b0ffa5+2023-04-27-22h-56m-58s",
    "PennPAL+c5f808b7+2023-06-15-17h-08m-12s",
    "PennPAL+c5f808b7+2023-10-09-21h-55m-24s",
    "RAIL+80edfcb1+2023-07-05-18h-22m-44s",
    "RAIL+80edfcb1+2023-07-13-20h-13m-59s",
    "RAIL+80edfcb1+2023-07-13-20h-54m-54s",
    "RAIL+80edfcb1+2023-07-14-17h-07m-42s",
    "RAIL+d027f2ae+2023-12-02-17h-29m-43s",
    "RAIL+t3d58310+2023-07-13-16h-01m-17s",
    "REAL+4dbb5646+2023-07-25-14h-12m-17s",
    "REAL+4f8ca688+2023-07-04-14h-45m-16s",
    "REAL+4f8ca688+2023-08-15-14h-31m-23s",
    "RPL+cb4f6842+2023-05-25-11h-51m-38s",
    "TRI+52ca9b6a+2023-11-21-17h-32m-03s",
    "TRI+52ca9b6a+2023-12-13-15h-12m-57s",
    "TRI+52ca9b6a+2024-01-03-16h-59m-30s",
    "TRI+52ca9b6a+2024-01-08-10h-46m-48s",
    "TRI+52ca9b6a+2024-01-16-14h-41m-19s",
    "TRI+749baf62+2023-09-06-13h-55m-06s",
    "TRI+749baf62+2023-09-06-13h-56m-42s",
    "WEIRD+e604eeed+2023-11-21-19h-26m-17s",
    "tri+7dfa2da3+2023-10-12-13h-13m-53s",
]
# The light arrays, fetched in full. Depth and masks are read lazily instead.
LIGHT_FILES = ["tracks_xyz.npy", "queries_xytv.npy"]
LIGHT_VIEW_FILES = ["intrinsics.npy", "extrinsics_w2c.npy", "visibility.npy",
                    "images_jpeg_bytes.npy"]
HEAVY_VIEW_FILES = ["depth.npy", "foreground_mask.npy"]

# Only the wrist camera ships a foreground mask; the two exterior views do not.
MASK_VIEWS = (0,)


def http_get(url, start=None, end=None, retries=4):
    """GET a URL, or one byte range of it, with a few retries."""
    headers = {} if start is None else {"Range": f"bytes={start}-{end}"}
    request = urllib.request.Request(url, headers=headers)
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                return response.read()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))


def fetch(relative_path, verbose=True):
    """Download one file of the sequence tree and cache it under DATA_ROOT."""
    out = DATA_ROOT / relative_path
    if out.exists():
        return out
    out.parent.mkdir(parents=True, exist_ok=True)
    payload = http_get(f"{BUCKET}/{relative_path}")
    partial = out.with_name(out.name + ".part")
    partial.write_bytes(payload)
    partial.rename(out)
    if verbose:
        # Episode names are long and all alike; only the view/file tail is useful.
        print(f"  {'/'.join(relative_path.split('/')[1:]):28s} {len(payload) / 1e6:8.1f} MB")
    return out


def view_files(view, with_depth):
    """Which files exist for a view: only the wrist camera has a mask."""
    names = list(LIGHT_VIEW_FILES)
    if with_depth:
        names += [name for name in HEAVY_VIEW_FILES
                  if name != "foreground_mask.npy" or view in MASK_VIEWS]
    return names


def fetch_sequence(sequence, views=(0, 1, 2), with_depth=False):
    """Fetch everything the notebook needs, in parallel."""
    wanted = [f"{sequence}/{name}" for name in LIGHT_FILES]
    for view in views:
        wanted += [f"{sequence}/{view}/{name}" for name in view_files(view, with_depth)]
    todo = [path for path in wanted if not (DATA_ROOT / path).exists()]
    print(f"{sequence}\n  {len(wanted) - len(todo)} files cached, downloading {len(todo)}")
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(fetch, todo))
    return DATA_ROOT / sequence


class RemoteFrames:
    """Frame-by-frame reader for a remote .npy, over HTTP range requests.

    DROID depth is 1280x720 float32 -- 3.7 MB a frame and up to 700 MB a view,
    while a notebook only ever looks at a handful of frames. The .npy header is
    parsed once from the first 4 KB, and each frame is then a single byte range.
    """

    def __init__(self, url, cache_size=8):
        header = io.BytesIO(http_get(url, 0, 4095))
        version = np.lib.format.read_magic(header)
        readers = {(1, 0): np.lib.format.read_array_header_1_0,
                   (2, 0): np.lib.format.read_array_header_2_0}
        self.shape, fortran, self.dtype = readers[version](header)
        assert not fortran, "Fortran-ordered .npy is not supported here"
        self.url = url
        self.offset = header.tell()
        self.frame_bytes = int(np.prod(self.shape[1:])) * self.dtype.itemsize
        self.cache_size = cache_size
        self._cache = {}

    def __len__(self):
        return int(self.shape[0])

    def __getitem__(self, frame):
        frame = int(frame) % len(self)
        if frame not in self._cache:
            if len(self._cache) >= self.cache_size:
                self._cache.pop(next(iter(self._cache)))
            start = self.offset + frame * self.frame_bytes
            payload = http_get(self.url, start, start + self.frame_bytes - 1)
            self._cache[frame] = np.frombuffer(payload, dtype=self.dtype).reshape(self.shape[1:])
        return self._cache[frame]


def frame_stack(sequence, view, name):
    """A frame-indexable handle: a memmap if downloaded, otherwise remote."""
    local = DATA_ROOT / sequence / str(view) / name
    if local.exists():
        return np.load(local, mmap_mode="r")
    return RemoteFrames(f"{BUCKET}/{sequence}/{view}/{name}")


def decode_jpeg(raw):
    """Decode one frame's JPEG bytes to an RGB uint8 image."""
    buffer = np.frombuffer(bytes(raw), dtype=np.uint8)
    return cv2.cvtColor(cv2.imdecode(buffer, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def project_tracks(tracks_xyz, intrinsics, extrinsics_w2c):
    """Project world points through one view's cameras; returns (xy, z)."""
    points_h = np.concatenate(
        [tracks_xyz, np.ones((*tracks_xyz.shape[:2], 1), dtype=np.float32)], axis=-1)
    points_camera = np.einsum("tij,tnj->tni", extrinsics_w2c, points_h)
    z = points_camera[..., 2]
    with np.errstate(invalid="ignore", divide="ignore"):
        xy = points_camera[..., :2] / z[..., None]
    xy = xy * intrinsics[None, None, :2] + intrinsics[None, None, 2:]
    return xy.astype(np.float32), z.astype(np.float32)


def camera_centers(extrinsics_w2c):
    """World-space camera centres from world-to-camera transforms."""
    rotation, translation = extrinsics_w2c[:, :3, :3], extrinsics_w2c[:, :3, 3]
    return -np.einsum("tji,tj->ti", rotation, translation)


def cap_depth(depth):
    """Apply the benchmark's DROID depth policy: zero outside the 2 m workspace."""
    capped = np.asarray(depth, dtype=np.float32).copy()
    capped[~(np.isfinite(capped) & (capped > 0.0) & (capped <= MAX_TRACKING_DEPTH_M))] = 0.0
    return capped


@dataclass(eq=False)
class View:
    index: int
    intrinsics: np.ndarray        # (4,) fx, fy, cx, cy
    extrinsics_w2c: np.ndarray    # (T, 4, 4)
    visibility: np.ndarray        # (T, N)
    jpegs: np.ndarray             # (T,) object array of bytes
    depth_frames: object          # frame-indexable (T, H, W) float32
    mask_frames: object           # frame-indexable (T, H, W) bool, or None

    def image(self, frame):
        return decode_jpeg(self.jpegs[int(frame)])

    def depth(self, frame, *, capped=False):
        raw = np.asarray(self.depth_frames[int(frame)], dtype=np.float32)
        return cap_depth(raw) if capped else raw

    def foreground_mask(self, frame):
        if self.mask_frames is None:
            return None
        return np.asarray(self.mask_frames[int(frame)]).astype(bool)

    @property
    def has_mask(self):
        return self.mask_frames is not None

    @functools.cached_property
    def centers(self):
        return camera_centers(self.extrinsics_w2c)

    @functools.cached_property
    def camera_motion_m(self):
        return float(np.linalg.norm(self.centers - self.centers[0], axis=-1).max())

    @property
    def is_static(self):
        return self.camera_motion_m < 0.01

    @property
    def kind(self):
        """DROID rigs are one wrist camera on the arm plus two fixed exteriors.

        View 0 is always the wrist camera -- it is the only view that ever moves
        and the only one shipping a mask -- but in a few episodes the arm barely
        translates, so the label reports the measured motion rather than
        inferring the mount from it.
        """
        if self.index in MASK_VIEWS:
            return f"wrist, moves {self.camera_motion_m:.2f}m"
        return "fixed exterior"

    @functools.cached_property
    def image_hw(self):
        return self.image(0).shape[:2]


@dataclass(eq=False)
class Sequence:
    name: str
    tracks_xyz: np.ndarray        # (T, N, 3) world coordinates
    queries_xytv: np.ndarray      # (N, 4) x, y, t, view
    views: list

    @property
    def num_frames(self):
        return self.tracks_xyz.shape[0]

    @property
    def num_tracks(self):
        return self.tracks_xyz.shape[1]

    @property
    def num_views(self):
        return len(self.views)

    @functools.cached_property
    def visibility(self):
        return np.stack([view.visibility for view in self.views], axis=-1)  # (T, N, V)

    @property
    def query_t(self):
        return self.queries_xytv[:, 2].astype(int)

    @property
    def query_v(self):
        return self.queries_xytv[:, 3].astype(int)

    @functools.lru_cache(maxsize=8)
    def project(self, view):
        data = self.views[view]
        return project_tracks(self.tracks_xyz, data.intrinsics, data.extrinsics_w2c)


def load_sequence(sequence, views=(0, 1, 2), with_depth=False):
    """Fetch (if needed) and open one DROID sequence."""
    root = fetch_sequence(sequence, views=views, with_depth=with_depth)
    return Sequence(
        name=sequence,
        tracks_xyz=np.load(root / "tracks_xyz.npy"),
        queries_xytv=np.load(root / "queries_xytv.npy"),
        views=[
            View(index=view,
                 intrinsics=np.load(root / str(view) / "intrinsics.npy"),
                 extrinsics_w2c=np.load(root / str(view) / "extrinsics_w2c.npy"),
                 visibility=np.load(root / str(view) / "visibility.npy"),
                 jpegs=np.load(root / str(view) / "images_jpeg_bytes.npy", allow_pickle=True),
                 depth_frames=frame_stack(sequence, view, "depth.npy"),
                 mask_frames=(frame_stack(sequence, view, "foreground_mask.npy")
                              if view in MASK_VIEWS else None))
            for view in views
        ])


In [ ]:
#@title Drawing and statistics helpers (run once)
def track_colors(track_ids, colormap="turbo"):
    """One stable RGB colour per drawn track, (len(ids), 3) uint8."""
    fractions = np.linspace(0.05, 0.95, max(len(np.asarray(track_ids)), 1))
    cmap = matplotlib.colormaps[colormap]
    return (np.array([cmap(f)[:3] for f in fractions]) * 255).astype(np.uint8)


def view_color(view):
    """The per-camera colour used for frustums and trajectories."""
    palette = np.array([[228, 92, 74], [74, 160, 228], [96, 200, 110], [220, 170, 60]])
    return palette[view % len(palette)]


def pick_tracks(sequence, count=40, *, frame=None, require_views=1,
                min_motion_m=0.0, seed=72):
    """Sample track ids worth drawing.

    ``require_views`` keeps only tracks that at least that many cameras ever
    see, which is how the cross-view panels get points that exist in more than
    one image; ``min_motion_m`` drops the roughly half of DROID tracks that sit
    on the static table and scene, leaving the arm and the manipulated objects.
    """
    visibility = sequence.visibility
    eligible = visibility.any(axis=0).sum(axis=-1) >= require_views
    if frame is not None:
        eligible &= visibility[frame].any(axis=-1)
    if min_motion_m > 0:
        path = np.linalg.norm(np.diff(sequence.tracks_xyz, axis=0), axis=-1).sum(axis=0)
        eligible &= path >= min_motion_m
    candidates = np.flatnonzero(eligible)
    if len(candidates) <= count:
        return candidates
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(candidates, size=count, replace=False))


def draw_points(image, xy, *, visible=None, colors=None, radius=4):
    """Draw projected points: filled when visible, hollow when occluded."""
    canvas = np.ascontiguousarray(image.copy())
    height, width = canvas.shape[:2]
    colors = track_colors(np.arange(len(xy))) if colors is None else colors
    for index, point in enumerate(xy):
        if not np.isfinite(point).all():
            continue
        x, y = int(round(float(point[0]))), int(round(float(point[1])))
        if not (-radius <= x < width + radius and -radius <= y < height + radius):
            continue
        color = tuple(int(c) for c in colors[index % len(colors)])
        is_visible = True if visible is None else bool(visible[index])
        cv2.circle(canvas, (x, y), radius, color, -1 if is_visible else 1, cv2.LINE_AA)
        if is_visible:
            cv2.circle(canvas, (x, y), radius, (255, 255, 255), 1, cv2.LINE_AA)
    return canvas


def draw_trails(canvas, trail_xy, colors, *, valid=None):
    """Draw per-track trails from (L, N, 2) positions in the current frame."""
    length = trail_xy.shape[0]
    for index in range(trail_xy.shape[1]):
        color = tuple(int(c) for c in colors[index % len(colors)])
        for step in range(1, length):
            start, end = trail_xy[step - 1, index], trail_xy[step, index]
            if not (np.isfinite(start).all() and np.isfinite(end).all()):
                continue
            if valid is not None and not (valid[step - 1, index] and valid[step, index]):
                continue
            cv2.line(canvas,
                     (int(round(float(start[0]))), int(round(float(start[1])))),
                     (int(round(float(end[0]))), int(round(float(end[1])))),
                     color, max(1, int(round(2.0 * step / length))), cv2.LINE_AA)
    return canvas


def label_panel(image, text):
    """Stamp a caption in the top-left corner of a panel."""
    canvas = np.ascontiguousarray(image)
    scale = max(0.5, canvas.shape[1] / 900.0)
    origin = (int(8 * scale), int(28 * scale))
    for color, thickness in (((0, 0, 0), int(4 * scale)),
                             ((255, 255, 255), max(1, int(1.5 * scale)))):
        cv2.putText(canvas, text, origin, cv2.FONT_HERSHEY_SIMPLEX, 0.7 * scale,
                    color, thickness, cv2.LINE_AA)
    return canvas


def letterbox(image, cell_hw, background=(0, 0, 0)):
    """Fit an image into a cell, preserving aspect ratio."""
    cell_h, cell_w = cell_hw
    height, width = image.shape[:2]
    scale = min(cell_w / width, cell_h / height)
    resized = cv2.resize(image, (max(1, int(width * scale)), max(1, int(height * scale))),
                         interpolation=cv2.INTER_AREA)
    canvas = np.full((cell_h, cell_w, 3), np.array(background, dtype=np.uint8), dtype=np.uint8)
    top, left = (cell_h - resized.shape[0]) // 2, (cell_w - resized.shape[1]) // 2
    canvas[top:top + resized.shape[0], left:left + resized.shape[1]] = resized
    return canvas


def montage(panels, *, columns=None, cell_width=480, background=(0, 0, 0)):
    """Tile panels of mixed sizes into one grid image."""
    assert panels
    columns = columns or min(len(panels), int(np.ceil(np.sqrt(len(panels)))))
    rows = int(np.ceil(len(panels) / columns))
    aspect = max(panel.shape[0] / panel.shape[1] for panel in panels)
    cell_hw = (int(cell_width * aspect), cell_width)
    cells = [letterbox(panel, cell_hw, background) for panel in panels]
    blank = np.full((*cell_hw, 3), np.array(background, dtype=np.uint8), dtype=np.uint8)
    cells += [blank] * (rows * columns - len(cells))
    return np.vstack([np.hstack(cells[r * columns:(r + 1) * columns]) for r in range(rows)])


def view_panel(sequence, view, frame, track_ids, *, colors=None, trail_length=0, label=True):
    """One view's frame with the ground-truth points drawn on it.

    Trails project *past* world positions through the *current* frame's camera,
    so a world-static point leaves no trail even though the camera is moving —
    trail length reads as world motion, not as camera motion.
    """
    data = sequence.views[view]
    canvas = data.image(frame)
    colors = track_colors(track_ids) if colors is None else colors
    if trail_length > 1:
        start = max(0, frame - trail_length + 1)
        trail_xyz = sequence.tracks_xyz[start:frame + 1, track_ids]
        extrinsics = np.repeat(data.extrinsics_w2c[frame][None], len(trail_xyz), axis=0)
        trail_xy, trail_z = project_tracks(trail_xyz, data.intrinsics, extrinsics)
        canvas = draw_trails(canvas, trail_xy, colors, valid=trail_z > 1e-3)
    xy, z = sequence.project(view)
    visible = data.visibility[frame, track_ids] & (z[frame, track_ids] > 0)
    canvas = draw_points(canvas, xy[frame, track_ids], visible=visible, colors=colors,
                         radius=max(3, int(round(min(canvas.shape[:2]) / 160))))
    if label:
        canvas = label_panel(canvas, f"view {view} ({data.kind})"
                                     f"  frame {frame}/{sequence.num_frames - 1}"
                                     f"  vis {int(visible.sum())}/{len(track_ids)}")
    return canvas


def multiview_grid(sequence, frame, track_ids, *, trail_length=0, cell_width=480, columns=None):
    """Every view at one frame, same tracks in the same colours."""
    colors = track_colors(track_ids)
    panels = [view_panel(sequence, view, frame, track_ids, colors=colors,
                         trail_length=trail_length) for view in range(sequence.num_views)]
    return montage(panels, columns=columns, cell_width=cell_width)


def crossview_patches(sequence, track, frame, *, patch=112, out_size=180):
    """Crop the same 3D point out of every view at one frame."""
    panels = []
    for view in range(sequence.num_views):
        data = sequence.views[view]
        xy, z = sequence.project(view)
        point = xy[frame, track]
        image = data.image(frame)
        height, width = image.shape[:2]
        half = patch // 2
        x = int(round(float(np.clip(point[0], half, width - half - 1))))
        y = int(round(float(np.clip(point[1], half, height - half - 1))))
        crop = cv2.resize(image[y - half:y + half, x - half:x + half],
                          (out_size, out_size), interpolation=cv2.INTER_NEAREST)
        scale = out_size / patch
        center = (int(round((float(point[0]) - (x - half)) * scale)),
                  int(round((float(point[1]) - (y - half)) * scale)))
        visible = bool(data.visibility[frame, track]) and float(z[frame, track]) > 0
        color = (60, 220, 60) if visible else (220, 60, 60)
        cv2.drawMarker(crop, center, color, cv2.MARKER_CROSS, int(out_size * 0.25), 2, cv2.LINE_AA)
        cv2.rectangle(crop, (0, 0), (out_size - 1, out_size - 1), color, 3)
        panels.append(label_panel(crop, f"v{view} {'vis' if visible else 'occl'}"
                                        f" z={float(z[frame, track]):.2f}"))
    return panels


def colorize_depth(depth, *, percentile=99.0):
    """Colour one depth frame over its own valid range; invalid pixels stay black."""
    valid = np.isfinite(depth) & (depth > 0.0)
    canvas = np.zeros((*depth.shape, 3), dtype=np.uint8)
    if not valid.any():
        return canvas
    low = float(np.percentile(depth[valid], 100.0 - percentile))
    high = float(np.percentile(depth[valid], percentile))
    high = high if high > low else low + 1e-3
    with np.errstate(invalid="ignore"):
        normalized = np.clip((depth - low) / (high - low), 0.0, 1.0)
    scaled = np.nan_to_num(normalized * 255.0, nan=0.0, posinf=255.0, neginf=0.0)
    colored = cv2.applyColorMap(scaled.astype(np.uint8), cv2.COLORMAP_TURBO)
    canvas[valid] = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)[valid]
    return canvas


def overlay_mask(image, mask, *, alpha=0.45, color=(255, 64, 64)):
    """Tint the masked pixels of an image."""
    canvas = image.astype(np.float32).copy()
    canvas[mask] = (1.0 - alpha) * canvas[mask] + alpha * np.array(color, dtype=np.float32)
    return canvas.astype(np.uint8)


def unproject_depth(depth, image, intrinsics, extrinsics_w2c, *, stride=8,
                    max_depth_m=MAX_TRACKING_DEPTH_M):
    """Lift one depth frame to a world point cloud; returns (points, colors)."""
    height, width = depth.shape
    rows, columns = np.mgrid[0:height:stride, 0:width:stride]
    z = depth[::stride, ::stride]
    valid = np.isfinite(z) & (z > 0.0)
    if max_depth_m is not None:
        valid &= z <= max_depth_m
    fx, fy, cx, cy = [float(value) for value in intrinsics]
    points_camera = np.stack([(columns - cx) / fx * z, (rows - cy) / fy * z, z], axis=-1)[valid]
    rotation, translation = extrinsics_w2c[:3, :3], extrinsics_w2c[:3, 3]
    points_world = (points_camera - translation) @ rotation
    scale_h, scale_w = image.shape[0] / height, image.shape[1] / width
    color_rows = np.clip((rows * scale_h).astype(np.int64), 0, image.shape[0] - 1)
    color_columns = np.clip((columns * scale_w).astype(np.int64), 0, image.shape[1] - 1)
    return points_world.astype(np.float32), image[color_rows, color_columns][valid]


def sample_depth_at(depth, xy):
    """Read a depth map at projected pixel coordinates; NaN outside the raster."""
    height, width = depth.shape
    columns, rows = np.floor(xy[..., 0] + 0.5), np.floor(xy[..., 1] + 0.5)
    inside = (np.isfinite(columns) & np.isfinite(rows) & (columns >= 0) & (columns < width)
              & (rows >= 0) & (rows < height))
    out = np.full(xy.shape[:-1], np.nan, dtype=np.float32)
    out[inside] = depth[rows[inside].astype(np.int64), columns[inside].astype(np.int64)]
    return out


def covisibility_matrix(visibility):
    """(V, V): the fraction of observations seen by view i and view j at once."""
    flat = visibility.reshape(-1, visibility.shape[-1]).astype(np.float32)
    return (flat.T @ flat) / flat.shape[0]


def hex_colors(colors):
    """Vectorized #rrggbb strings for plotly, one per point."""
    colors = np.asarray(colors, dtype=np.uint8).reshape(-1, 3)
    packed = (colors[:, 0].astype(np.uint32) << 16 | colors[:, 1].astype(np.uint32) << 8
              | colors[:, 2].astype(np.uint32))
    return ["#%06x" % value for value in packed]


In [ ]:
#@title Plotly 3D scene helper (run once)
def plot_scene_3d(sequence, frame, track_ids, *, trail_length=30, cloud_views=(1, 2),
                  cloud_stride=6, max_cloud_points=40_000, max_depth_m=MAX_TRACKING_DEPTH_M,
                  colors=None, cloud_size=1.6, point_size=4.0, title=None, height=720):
    """The world at one frame: depth cloud, camera trajectories, and 3D tracks.

    Each camera is drawn as its whole trajectory plus a marker at ``frame``, so
    the four moving DIEGESIS cameras read as arcs rather than dots. Trails go
    into one NaN-separated trace and the heads into another — a few traces
    serialize and render far faster than two per track.
    """
    import plotly.graph_objects as go

    figure = go.Figure()
    for view in cloud_views:
        data = sequence.views[view]
        points, point_colors = unproject_depth(
            data.depth(frame), data.image(frame), data.intrinsics,
            data.extrinsics_w2c[frame], stride=cloud_stride, max_depth_m=max_depth_m)
        if not len(points):
            continue
        if len(points) > max_cloud_points:
            keep = np.random.default_rng(72).choice(len(points), max_cloud_points, replace=False)
            points, point_colors = points[keep], point_colors[keep]
        figure.add_trace(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
            marker=dict(size=cloud_size, color=hex_colors(point_colors)),
            name=f"cloud view {view}", hoverinfo="skip"))

    for view, data in enumerate(sequence.views):
        centers = data.centers
        line_color = "rgb({},{},{})".format(*view_color(view))
        figure.add_trace(go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="lines",
            line=dict(width=3, color=line_color), name=f"camera {view} path", hoverinfo="skip"))
        figure.add_trace(go.Scatter3d(
            x=[centers[frame, 0]], y=[centers[frame, 1]], z=[centers[frame, 2]],
            mode="markers+text", marker=dict(size=5, symbol="diamond", color=line_color),
            text=[f"v{view}"], textposition="top center", name=f"camera {view}"))

    track_ids = np.asarray(track_ids)
    colors = track_colors(track_ids) if colors is None else np.asarray(colors)
    start = max(0, frame - trail_length + 1)
    trails = sequence.tracks_xyz[start:frame + 1, track_ids]                 # (L, N, 3)
    separator = np.full((1, len(track_ids), 3), np.nan, dtype=np.float32)
    joined = np.concatenate([trails, separator]).transpose(1, 0, 2).reshape(-1, 3)
    figure.add_trace(go.Scatter3d(
        x=joined[:, 0], y=joined[:, 1], z=joined[:, 2], mode="lines",
        line=dict(color=np.repeat(hex_colors(colors), trails.shape[0] + 1).tolist(), width=4),
        name="trails", hoverinfo="skip"))
    heads = sequence.tracks_xyz[frame, track_ids]
    figure.add_trace(go.Scatter3d(
        x=heads[:, 0], y=heads[:, 1], z=heads[:, 2], mode="markers",
        marker=dict(color=hex_colors(colors), size=point_size),
        text=[f"track {int(track)}" for track in track_ids], hoverinfo="text", name="tracks"))

    figure.update_layout(
        height=height, margin=dict(l=0, r=0, t=30, b=0),
        title=title if title is not None else f"droid/{sequence.name} — frame {frame}",
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"))
    return figure


## 2. Pick a sequence

The 50 evaluated episodes come from several DROID labs (`AUTOLab`, `IRIS`, `TRI`, …). Pick by
index; the download is cached under `droid_data/`, so coming back to one is instant.


In [ ]:
SEQUENCE_INDEX = 0        #@param {type:"slider", min:0, max:49, step:1}
# True fetches the full depth + masks; False reads them per frame over HTTP ranges.
DOWNLOAD_DEPTH = False    #@param {type:"boolean"}

SEQUENCE = SEQUENCES[SEQUENCE_INDEX]
FRAME = None              # None = the midpoint of the episode
NUM_TRACKS = 24           # how many tracks to draw
TRAIL = 12                # trail length, in frames
REQUIRE_VIEWS = 2         # only draw points that at least this many cameras ever see
MIN_MOTION_M = 0.0        # >0 keeps only moving points (arm and manipulated objects)

sequence = load_sequence(SEQUENCE, with_depth=DOWNLOAD_DEPTH)
FRAME = sequence.num_frames // 2 if FRAME is None else FRAME
track_ids = pick_tracks(sequence, NUM_TRACKS, frame=FRAME,
                        require_views=REQUIRE_VIEWS, min_motion_m=MIN_MOTION_M)
colors = track_colors(track_ids)

visibility = sequence.visibility
observed = visibility.sum(axis=-1)
path_m = np.linalg.norm(np.diff(sequence.tracks_xyz, axis=0), axis=-1).sum(axis=0)

print(f"droid/{SEQUENCE}  [{SEQUENCE_INDEX}/{len(SEQUENCES) - 1}]")
print(f"  {sequence.num_frames} frames x {sequence.num_tracks} tracks x {sequence.num_views} views"
      f"  |  images {sequence.views[0].image_hw}")
print(f"  visible rate {visibility.mean():.3f}"
      f"  |  observations seen by >=2 views {(observed >= 2).sum() / max(1, (observed >= 1).sum()):.3f}"
      f"  |  mean views per track {visibility.any(axis=0).sum(axis=-1).mean():.2f}")
print(f"  world-static tracks (<2 cm of path) {(path_m < 0.02).mean():.1%}"
      f"  |  workspace extent (m) {np.round(np.ptp(sequence.tracks_xyz.reshape(-1, 3), axis=0), 2).tolist()}")
print(f"  drawing {len(track_ids)} tracks at frame {FRAME}")


## 3. The rig: one moving camera, two fixed ones

This is DROID's signature. View 0 is mounted on the wrist and travels with the arm; views 1 and 2
are bolted to the bench. The table below is the whole story, and the plot shows the wrist camera's
path — which is, up to the mount offset, **the trajectory of the robot's end effector**.

(In a handful of the 50 episodes the arm barely translates, so view 0's motion is near zero too.)


In [ ]:
rows = []
for view, data in enumerate(sequence.views):
    rows.append(dict(view=view, kind=data.kind, image_hw=f"{data.image_hw[0]}x{data.image_hw[1]}",
                     fx=round(float(data.intrinsics[0]), 1),
                     cx=round(float(data.intrinsics[2]), 1),
                     motion_m=round(data.camera_motion_m, 3),
                     has_mask=data.has_mask,
                     visible_rate=round(float(data.visibility.mean()), 3)))
pd.DataFrame(rows)


In [ ]:
fig = plt.figure(figsize=(11, 4.4))

ax = fig.add_subplot(1, 2, 1, projection="3d")
for view, data in enumerate(sequence.views):
    centers = data.centers
    color = view_color(view) / 255.0
    if data.is_static:
        ax.scatter(*centers[0], color=color, s=60, marker="D", label=f"view {view} (fixed)")
    else:
        ax.plot(*centers.T, color=color, lw=2, label=f"view {view} (wrist)")
points = sequence.tracks_xyz[FRAME]
ax.scatter(*points.T, s=1, c="0.6", alpha=0.4)
ax.set(title="camera centres + the tracked points", xlabel="x", ylabel="y")
ax.legend(fontsize=8)

ax = fig.add_subplot(1, 2, 2)
for view, data in enumerate(sequence.views):
    ax.plot(np.linalg.norm(data.centers - data.centers[0], axis=-1),
            color=view_color(view) / 255.0, lw=1.8, label=f"view {view}")
ax.set(title="camera displacement from frame 0", xlabel="frame", ylabel="metres")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()


## 4. One frame, every view

The same 3D points in the same colours, projected into all three cameras. **Filled = visible in
that view, hollow = occluded or off-frame.**

The trail is *past world positions* seen through the *current* frame's camera. In the wrist view
that means a point on the table has **no** trail even though its pixel is racing across the image —
the trail measures world motion, never camera motion.


In [ ]:
grid = multiview_grid(sequence, FRAME, track_ids, trail_length=TRAIL, cell_width=620, columns=3)
plt.figure(figsize=(16, 16 * grid.shape[0] / grid.shape[1]))
plt.imshow(grid)
plt.axis("off")
plt.title(f"droid/{SEQUENCE} — frame {FRAME}")
plt.show()


## 5. The whole episode as a three-view video


In [ ]:
VIDEO_FRAMES = 120     # cap; beyond this the frames are sampled uniformly
frames = np.unique(np.linspace(0, sequence.num_frames - 1,
                               min(VIDEO_FRAMES, sequence.num_frames)).astype(int))

video = np.stack([multiview_grid(sequence, int(frame), track_ids,
                                 trail_length=TRAIL, cell_width=380, columns=3)
                  for frame in frames])
mediapy.show_video(video, fps=12, title=f"droid/{SEQUENCE} — all three views")


## 6. Multi-view structure

### 6.1 Co-visibility

The diagonal is each view's own visible rate; entry `(i, j)` is the fraction of observations views
`i` and `j` see at the same instant. DROID's three cameras all point at one small tabletop, so this
matrix is **much fuller than DIEGESIS's** — around 70% of observations are seen by two or more
cameras. That is what makes it the benchmark's easiest cross-view source, and the wrist view is the
one that breaks the symmetry by diving in close.


In [ ]:
covis = covisibility_matrix(visibility)

fig, ax = plt.subplots(figsize=(3.8, 3.8))
image = ax.imshow(covis, vmin=0, vmax=covis.max(), cmap="viridis")
for i in range(sequence.num_views):
    for j in range(sequence.num_views):
        ax.text(j, i, f"{covis[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="w" if covis[i, j] < covis.max() * 0.6 else "k")
ax.set(xticks=range(sequence.num_views), yticks=range(sequence.num_views),
       xlabel="view", ylabel="view", title="co-visibility")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()


In [ ]:
per_track = visibility.any(axis=0).sum(axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].hist(per_track, bins=np.arange(sequence.num_views + 2) - 0.5, color="#4c72b0")
axes[0].set(title="views that ever see a track", xlabel="#views", ylabel="tracks")
axes[1].hist(observed.ravel(), bins=np.arange(sequence.num_views + 2) - 0.5, color="#dd8452")
axes[1].set(title="views seeing an observation", xlabel="#views", ylabel="observations")
axes[2].plot(observed.mean(axis=1), lw=1.5)
axes[2].set(title="mean co-visible views over time", xlabel="frame", ylabel="#views")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()


### 6.2 The same point in every camera

One 3D identity, one instant, one crop per camera: green border = that view calls it visible, red =
occluded or off-frame. On DROID the wrist crop is usually far more zoomed-in than the exterior
ones, which is the cross-view scale change a tracker has to survive.


In [ ]:
visible_counts = visibility[FRAME][track_ids].sum(axis=-1)
FOCUS_TRACK = int(track_ids[int(np.argmax(visible_counts))])

strip = montage(crossview_patches(sequence, FOCUS_TRACK, FRAME, patch=140, out_size=200),
                columns=sequence.num_views, cell_width=200)
plt.figure(figsize=(2.2 * sequence.num_views, 2.6))
plt.imshow(strip)
plt.axis("off")
plt.title(f"track {FOCUS_TRACK} @ frame {FRAME} — the same point in every camera")
plt.show()

print("projected pixel coordinates per view:")
for view in range(sequence.num_views):
    xy, z = sequence.project(view)
    state = "visible" if visibility[FRAME, FOCUS_TRACK, view] else "occluded / out of frame"
    print(f"  view {view} ({sequence.views[view].kind:22s}):"
          f" ({xy[FRAME, FOCUS_TRACK, 0]:8.2f}, {xy[FRAME, FOCUS_TRACK, 1]:8.2f})"
          f"  z={z[FRAME, FOCUS_TRACK]:5.2f} m  {state}")


### 6.3 Where the queries live

The fourth column of `queries_xytv` is the **query view**. The benchmark's `query_view` /
`non_query_view` buckets come from here: a monocular method can only answer in the camera it was
queried in, and only a multi-view method scores on the other two.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))

axes[0].bar(range(sequence.num_views),
            np.bincount(sequence.query_v, minlength=sequence.num_views), color="#4c72b0")
axes[0].set(title="queries per view", xlabel="view", ylabel="tracks")
axes[1].hist(sequence.query_t, bins=min(40, sequence.num_frames), color="#dd8452")
axes[1].set(title="query frame distribution", xlabel="frame", ylabel="tracks")
scatter = axes[2].scatter(sequence.queries_xytv[:, 0], sequence.queries_xytv[:, 1],
                          c=sequence.query_v, s=5, cmap="tab10")
axes[2].invert_yaxis()
axes[2].set(title="query pixel positions (all views overlaid)", xlabel="x", ylabel="y")
fig.colorbar(scatter, ax=axes[2], label="query view")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()


In [ ]:
# The queries themselves, drawn on the view and frame they were annotated in.
QUERY_VIEW = int(np.bincount(sequence.query_v, minlength=sequence.num_views).argmax())
owned = np.flatnonzero(sequence.query_v == QUERY_VIEW)
query_frame = int(np.bincount(sequence.query_t[owned]).argmax())
on_frame = owned[sequence.query_t[owned] == query_frame]

panel = draw_points(sequence.views[QUERY_VIEW].image(query_frame),
                    sequence.queries_xytv[on_frame, :2],
                    colors=track_colors(on_frame), radius=5)
plt.figure(figsize=(10, 10 * panel.shape[0] / panel.shape[1]))
plt.imshow(panel)
plt.axis("off")
plt.title(f"{len(on_frame)} queries on view {QUERY_VIEW}"
          f" ({sequence.views[QUERY_VIEW].kind}), frame {query_frame}")
plt.show()


## 7. Depth, the 2 m workspace policy, and the wrist mask

Three DROID-specific things at once:

- **stereo depth is noisy and partial** — expect holes on specular and textureless surfaces,
  unlike DIEGESIS's perfect synthetic depth;
- **the 2 m cap** is what the benchmark actually feeds a tracker: everything past the workspace is
  zeroed, which on the exterior views removes the far wall and the rest of the lab;
- **only view 0 has a foreground mask**, so the panel for it appears and the exterior views say so.


In [ ]:
DEPTH_VIEW = 1     # 0 = wrist, 1 and 2 = fixed exterior

data = sequence.views[DEPTH_VIEW]
image = data.image(FRAME)
raw_depth = data.depth(FRAME)
capped_depth = data.depth(FRAME, capped=True)
raw_valid = np.isfinite(raw_depth) & (raw_depth > 0)
kept = capped_depth > 0

panels = [("RGB", image),
          ("raw depth (turbo)", colorize_depth(raw_depth)),
          (f"valid raw depth ({raw_valid.mean():.1%})",
           overlay_mask(image, raw_valid, color=(64, 200, 64))),
          (f"kept by the 2 m policy ({kept.mean():.1%})",
           overlay_mask(image, kept, color=(64, 160, 255)))]

mask = data.foreground_mask(FRAME)
if mask is not None:
    panels.append((f"foreground_mask ({mask.mean():.1%})", overlay_mask(image, mask)))
else:
    print(f"view {DEPTH_VIEW} ships no foreground mask (only view 0 does)")

fig, axes = plt.subplots(1, len(panels), figsize=(4.0 * len(panels), 3.0))
for ax, (title, panel) in zip(np.atleast_1d(axes), panels):
    ax.imshow(panel)
    ax.set_title(f"view {DEPTH_VIEW} — {title}", fontsize=9)
    ax.axis("off")
fig.tight_layout()

print(f"raw depth on valid pixels: {raw_depth[raw_valid].min():.2f} – {raw_depth[raw_valid].max():.2f} m")
print(f"the 2 m cap discards {100 * (raw_valid.sum() - kept.sum()) / max(1, raw_valid.sum()):.1f}%"
      f" of the valid pixels")


In [ ]:
# How much each view loses to the cap, and how deep the tracked points actually are.
rows = []
for view, data in enumerate(sequence.views):
    depth = data.depth(FRAME)
    valid = np.isfinite(depth) & (depth > 0)
    _, z = sequence.project(view)
    seen = data.visibility[FRAME] & (z[FRAME] > 0)
    rows.append(dict(view=view, kind=data.kind,
                     valid_depth=round(float(valid.mean()), 3),
                     kept_by_2m=round(float((depth[valid] <= MAX_TRACKING_DEPTH_M).mean()), 3),
                     median_depth_m=round(float(np.median(depth[valid])), 2),
                     median_track_z_m=round(float(np.median(z[FRAME][seen])), 2) if seen.any() else None,
                     has_mask=data.has_mask))
pd.DataFrame(rows)


## 8. Is the ground truth self-consistent across views?

Read the **source view's depth map** at the ground-truth pixel, unproject into the world, project
into the **target view**, and compare with the target view's own ground-truth pixel. On DROID this
is a real test rather than a formality: the depth is stereo, not synthetic, so the error here is
the actual noise floor of the reconstruction — a few pixels rather than a hundredth of one.


In [ ]:
# The two views that share the most points on this frame.
pairs = {(i, j): (visibility[:, :, i] & visibility[:, :, j]).sum(axis=1)
         for i in range(sequence.num_views) for j in range(sequence.num_views) if i < j}
SOURCE_VIEW, TARGET_VIEW = max(pairs, key=lambda pair: pairs[pair][FRAME])
CHECK_FRAME = FRAME
if pairs[(SOURCE_VIEW, TARGET_VIEW)][FRAME] < 20:
    SOURCE_VIEW, TARGET_VIEW = max(pairs, key=lambda pair: pairs[pair].max())
    CHECK_FRAME = int(pairs[(SOURCE_VIEW, TARGET_VIEW)].argmax())
    print(f"frame {FRAME} has few co-visible points; checking frame {CHECK_FRAME} instead")

source, target = sequence.views[SOURCE_VIEW], sequence.views[TARGET_VIEW]
xy_source, z_source = sequence.project(SOURCE_VIEW)
xy_target, _ = sequence.project(TARGET_VIEW)
sampled = sample_depth_at(source.depth(CHECK_FRAME), xy_source[CHECK_FRAME])
usable = (source.visibility[CHECK_FRAME] & target.visibility[CHECK_FRAME]
          & np.isfinite(sampled) & (sampled > 0))

print(f"view {SOURCE_VIEW} depth -> world -> view {TARGET_VIEW} at frame {CHECK_FRAME}:"
      f" {int(usable.sum())} usable co-visible points")
assert usable.any(), "no co-visible point with valid depth anywhere in this episode"

fx, fy, cx, cy = [float(value) for value in source.intrinsics]
u, v = xy_source[CHECK_FRAME, usable].T
z = sampled[usable]
points_camera = np.stack([(u - cx) / fx * z, (v - cy) / fy * z, z], axis=-1)
extrinsics = source.extrinsics_w2c[CHECK_FRAME]
points_world = (points_camera - extrinsics[:3, 3]) @ extrinsics[:3, :3]

reprojected, _ = project_tracks(points_world[None].astype(np.float32),
                                target.intrinsics, target.extrinsics_w2c[CHECK_FRAME][None])
error = np.linalg.norm(reprojected[0] - xy_target[CHECK_FRAME, usable], axis=-1)

print(f"  reprojection error: median {np.median(error):.2f} px,"
      f" 90th pct {np.percentile(error, 90):.2f} px  (on 1280x720)")
print(f"  track depth vs sampled depth map: median difference"
      f" {np.median(np.abs(z - z_source[CHECK_FRAME, usable])):.4f} m")

plt.figure(figsize=(6, 3))
plt.hist(np.clip(error, 0, np.percentile(error, 99)), bins=50, color="#4c72b0")
plt.xlabel("reprojection error (px)")
plt.ylabel("points")
plt.grid(alpha=0.3)
plt.show()


## 9. The 3D scene

The two exterior depth maps unprojected into the world (capped at 2 m, so you get the workspace
and not the lab), the camera trajectories — two dots and one arc — and the 3D tracks. Drag to
rotate. The arc *is* the arm's path.


In [ ]:
CLOUD_VIEWS = (1, 2)      # the fixed exteriors see the whole workspace
SCENE_TRACKS = 120

scene_ids = pick_tracks(sequence, SCENE_TRACKS, frame=FRAME, require_views=1)
figure = plot_scene_3d(sequence, FRAME, scene_ids, trail_length=TRAIL * 3,
                       cloud_views=CLOUD_VIEWS, cloud_stride=6)
figure.show()


## 10. How the points move in the world

`tracks_xyz` is in world space, so the manipulated objects separate cleanly from the table and the
rest of the lab. Roughly half of a DROID episode's points never move — and unlike the actors in
DIEGESIS, the moving ones move slowly and in short bursts, which is what makes them easy to track
in 2D and hard to pin down in depth.


In [ ]:
displacement_m = np.linalg.norm(sequence.tracks_xyz[-1] - sequence.tracks_xyz[0], axis=-1)
speed = np.linalg.norm(np.diff(sequence.tracks_xyz, axis=0), axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].hist(path_m, bins=60, color="#4c72b0")
axes[0].set(title="path length per track (m)", yscale="log")
axes[1].hist(displacement_m, bins=60, color="#dd8452")
axes[1].set(title="first-to-last displacement (m)", yscale="log")
axes[2].plot(np.median(speed, axis=1), lw=1.5, label="median")
axes[2].plot(np.percentile(speed, 90, axis=1), lw=1, alpha=0.7, label="p90")
axes[2].set(title="per-frame displacement (m/frame)", xlabel="frame")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()

print(f"static tracks (<2 cm of path): {(path_m < 0.02).mean():.1%}")
print(f"largest path length: {path_m.max():.2f} m")


### 10.1 Only the moving points

The same frame with the static table dropped: what is left is the arm, the gripper, and whatever
it is manipulating. Rather than a fixed threshold, this takes the most mobile tracks that are
visible right now — a few of the 50 episodes barely move at all, and a threshold would leave those
panels empty.


In [ ]:
visible_now = np.flatnonzero(visibility[FRAME].any(axis=-1))
moving_ids = visible_now[np.argsort(path_m[visible_now])[::-1][:40]]
print(f"the {len(moving_ids)} most mobile tracks visible on this frame — path length"
      f" {path_m[moving_ids].min():.3f}–{path_m[moving_ids].max():.3f} m")

grid_moving = multiview_grid(sequence, FRAME, moving_ids, trail_length=TRAIL * 3,
                             cell_width=620, columns=3)
plt.figure(figsize=(16, 16 * grid_moving.shape[0] / grid_moving.shape[1]))
plt.imshow(grid_moving)
plt.axis("off")
plt.title(f"droid/{SEQUENCE} — moving tracks only, frame {FRAME}")
plt.show()


## 11. Every episode, raw video, no ground truth

Stands on its own — it needs only the three helper cells in section 1, and draws **nothing** on the
frames. It fetches `images_jpeg_bytes.npy` for the chosen episodes (~65 MB each) and plays each as
a three-view strip, so the whole data source can be watched in one go.

All 50 episodes is ~3.2 GB and a fair wait; `OVERVIEW_SEQUENCES` defaults to the first 12. Set it
to `SEQUENCES` for everything.


In [ ]:
OVERVIEW_SEQUENCES = SEQUENCES[:12]   # set to SEQUENCES for all 50 (~3.2 GB)
OVERVIEW_VIEWS = (0, 1, 2)
OVERVIEW_FRAMES = 40                  # frames per video, sampled uniformly
OVERVIEW_CELL = 200                   # pixels per view across the three-view strip


def fetch_images(sequences, views=OVERVIEW_VIEWS):
    """Download only images_jpeg_bytes.npy for a list of episodes."""
    wanted = [f"{name}/{view}/images_jpeg_bytes.npy" for name in sequences for view in views]
    todo = [path for path in wanted if not (DATA_ROOT / path).exists()]
    print(f"{len(wanted) - len(todo)}/{len(wanted)} view files cached, downloading {len(todo)}"
          f" (~{len(todo) * 22} MB)")
    with ThreadPoolExecutor(max_workers=12) as pool:
        list(pool.map(lambda path: fetch(path, verbose=False), todo))
    print("done")


def sequence_frames(sequence, views=OVERVIEW_VIEWS):
    """The JPEG object arrays of one episode, one per view."""
    return [np.load(DATA_ROOT / sequence / str(view) / "images_jpeg_bytes.npy",
                    allow_pickle=True) for view in views]


def sequence_strip(jpegs, frame, *, cell_width=OVERVIEW_CELL, label=None):
    """One frame of an episode as a strip of its views, no annotations."""
    panels = [decode_jpeg(view_jpegs[min(int(frame), len(view_jpegs) - 1)])
              for view_jpegs in jpegs]
    strip = montage(panels, columns=len(panels), cell_width=cell_width)
    return label_panel(strip, label) if label else strip


def sequence_video(sequence, *, views=OVERVIEW_VIEWS, num_frames=OVERVIEW_FRAMES,
                   cell_width=OVERVIEW_CELL, label=True):
    """A three-view video of one episode: raw frames, nothing drawn on them."""
    jpegs = sequence_frames(sequence, views)
    frames = np.unique(np.linspace(0, len(jpegs[0]) - 1, num_frames).astype(int))
    return np.stack([sequence_strip(jpegs, frame, cell_width=cell_width,
                                    label=sequence.split("+")[0] if label else None)
                     for frame in frames])


fetch_images(OVERVIEW_SEQUENCES)


### 11.1 One frame per episode


In [ ]:
tiles = []
for name in OVERVIEW_SEQUENCES:
    jpegs = sequence_frames(name)
    tiles.append(sequence_strip(jpegs, (len(jpegs[0]) - 1) // 2, cell_width=220, label=name[:40]))

sheet = montage(tiles, columns=2, cell_width=660, background=(24, 24, 24))
plt.figure(figsize=(15, 15 * sheet.shape[0] / sheet.shape[1]))
plt.imshow(sheet)
plt.axis("off")
plt.title(f"DROID — {len(OVERVIEW_SEQUENCES)} episodes x {len(OVERVIEW_VIEWS)} views")
plt.show()


### 11.2 Every episode as a video

Episodes differ in length, so each clip is resampled to `OVERVIEW_FRAMES` frames.


In [ ]:
videos = {name.split("+")[0] + "…" + name[-13:]: sequence_video(name)
          for name in OVERVIEW_SEQUENCES}
print(f"{len(videos)} videos, {sum(video.nbytes for video in videos.values()) / 1e6:.0f} MB in RAM")

mediapy.show_videos(videos, fps=12, columns=2, height=200)


## 12. Save the figures


In [ ]:
import imageio.v3 as iio

out_dir = Path("droid_figures")
out_dir.mkdir(exist_ok=True)
stem = SEQUENCE.replace("+", "_")

grid_path = out_dir / f"{stem}_grid_f{FRAME}.png"
video_path = out_dir / f"{stem}_multiview.mp4"
scene_path = out_dir / f"{stem}_scene3d.html"

iio.imwrite(grid_path, grid)
mediapy.write_video(video_path, video, fps=12)
figure.write_html(scene_path, include_plotlyjs="cdn")
for path in (grid_path, video_path, scene_path):
    print(f"{str(path):60s} {path.stat().st_size / 1e6:6.1f} MB")

try:
    from google.colab import files
    for path in (grid_path, video_path, scene_path):
        files.download(str(path))
except ImportError:
    print("not on Colab — the files are on the local disk")
